# Information Entropy of $m$-Blocks
## Generalising $H(\mathcal{X})$ from single characters to blocks of $m$ characters

> 📺 **[Watch the original lecture on YouTube](https://youtu.be/6Ijdrrp0DeQ)**

In [Part 2](../Video_02/Information_Entropy_Examples.ipynb) every calculation treated one
**single character** as the unit of information. This lecture generalises that: instead of reading
a message one character at a time, we group characters into **$m$-blocks** — $m$ consecutive
characters treated as a single new character — and apply the identical Shannon formula to the
larger alphabet that results.

The payoff is a relationship between the size of the block and the number of distinct states
available to it:

$$\Omega = U^{\,m}$$

and, following from it, a strikingly simple scaling law for maximum entropy:

$$H(\mathcal{X})^{(m)}_{\max} = m \cdot H(\mathcal{X})_{\max}$$

That symbol $\Omega$ is not an accident of notation — it is Boltzmann's symbol for the number of
**microstates** of a physical system, and the link is the thread this series follows.

> 🧭 **Navigation** · [← Part 2: Information Entropy — Worked Examples](../Video_02/Information_Entropy_Examples.ipynb) · **Part 3 (current)** · *Part 4: coming soon*

---

## 1. Recap — entropy of a set of single characters

For a message drawn from an alphabet $\mathcal{X}$ of $U$ **distinct** characters with probabilities
$\{p_1,\dots,p_U\}$, Parts 1 and 2 gave

$$H(\mathcal{X}) = -\sum_{j=1}^{U} p_j \log_2 p_j \quad \text{[bits per character]}$$

with the uniform-distribution ceiling

$$H(\mathcal{X})_{\max} = \log_2 U .$$

| Symbol | Meaning |
|---|---|
| $N$ | total number of characters in the message |
| $U$ | number of **distinct** single characters |
| $p_j$ | probability that character $j$ occurs |

Everything that follows reuses this formula unchanged. All we do is redefine *what counts as a
character*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def shannon_entropy(probabilities, base=2):
    """Shannon entropy H = -sum(p * log_b(p)).  Skips zero-probability terms."""
    p = np.asarray(probabilities, dtype=float)
    p = p[p > 0]
    assert np.isclose(p.sum(), 1.0), f"Probabilities must sum to 1 (got {p.sum()})"
    return -np.sum(p * np.log(p) / np.log(base))

def entropy_of_message(message, base=2):
    """Compute H(X) directly from a sequence of symbols (string or list)."""
    N = len(message)
    counts = Counter(message)
    probs = [c / N for c in counts.values()]
    return shannon_entropy(probs, base=base), counts

---
## 2. The $m$-block, and the slide $s$

An **$m$-block** is simply $m$ single characters combined to form one new composite character. The
block size $m$ is our choice:

- $m = 1$ — the special case of Parts 1 and 2: each block *is* a single character.
- $m = 2, 3, 4, \ldots$ — pairs, triples, quadruples of characters.
- $m = N$ — the entire message collapsed into one single block.

An $m$-block might be a pair of bits, a codon of three DNA bases, or a whole word of English.

### Sliding the window

To generate blocks we slide a window of width $m$ along the message, advancing by a **slide** $s$
each step. The lecture imposes the condition

$$1 \leq s \leq m$$

The upper bound matters: if $s > m$ then the window jumps further than its own width, and the
characters between one block and the next are **skipped entirely** — the blocks would no longer
cover the message.

The two ends of that range are the cases worth naming:

| Slide | Behaviour | Number of blocks from $N$ characters |
|---|---|---|
| $s = m$ | **non-overlapping** — blocks tile the message end to end | $\lfloor N/m \rfloor$ |
| $s = 1$ | **fully overlapping** — a sliding window over every position | $N - m + 1$ |

Non-overlapping is the cleaner construction conceptually. Fully overlapping extracts far more
blocks from the same message, which makes it the better estimator when data is limited — we use it
later for exactly that reason.

### Presenter's Notes — Defining $\Omega$, the slide $s$, and $H(\mathcal{X})^{(m)}$
![Board defining the m-block, the slide parameter s, and omega](Screenshots/Part_3_00_05_12.jpg)

*Screenshot at timestamp 00:05:12*

---

In [ ]:
def make_blocks(sequence, m, s=None):
    """Slide a width-m window along `sequence`, advancing by s each step.

    s defaults to m (non-overlapping). The lecture's condition is 1 <= s <= m;
    a larger slide would skip characters between consecutive blocks.
    """
    s = m if s is None else s
    if not 1 <= s <= m:
        raise ValueError(f"slide s must satisfy 1 <= s <= m (got s={s}, m={m})")
    seq = list(sequence)
    return [tuple(seq[i:i + m]) for i in range(0, len(seq) - m + 1, s)]

demo = [0, 1, 1, 0, 1, 0, 0, 1]
print(f"message (N = {len(demo)}): {demo}\n")
for m in (1, 2, 3, 4):
    non_overlap = make_blocks(demo, m)             # s = m
    overlap = make_blocks(demo, m, s=1)            # s = 1
    print(f"m = {m}")
    print(f"  s = m (non-overlapping): {[''.join(map(str, b)) for b in non_overlap]}")
    print(f"  s = 1 (overlapping):     {[''.join(map(str, b)) for b in overlap]}")

---
## 3. The central relationship: $\Omega = U^{\,m}$

Just as $U$ counts the distinct single characters, we need a symbol for the number of **distinct
$m$-block characters** available in the new set. The lecture calls it $\Omega$.

How large is $\Omega$? Each of the $m$ slots in a block can independently hold any of the $U$
single characters, so the count multiplies:

$$\underbrace{U \times U \times \cdots \times U}_{m \text{ slots}} = U^{\,m}$$

$$\boxed{\;\Omega = U^{\,m}\;}$$

This is the relationship linking the alphabet size $U$, the chosen block size $m$, and the number
of distinct block-characters $\Omega$ in the new set.

**Consistency check.** For $m = 1$ the formula gives $\Omega = U^1 = U$ — the block alphabet
collapses back to the single-character alphabet, exactly as it must. Part 2 was the $m=1$ special
case of this lecture all along.

Note that $\Omega$ counts the blocks that are *possible*, not the ones that actually *appear*. A
real message of finite length will typically contain only a fraction of them — a point that
returns in Section 8.

### Presenter's Notes — $\Omega = U^{m}$
![Board showing the boxed relationship omega equals U to the power m](Screenshots/Part_3_00_09_25.jpg)

*Screenshot at timestamp 00:09:25*

---

In [ ]:
from itertools import product

U = 2  # binary alphabet {0, 1}
print(f"Single-character alphabet: U = {U}\n")
print(f"{'m':>3}  {'Omega = U^m':>12}   distinct m-blocks")
print("-" * 62)
for m in range(1, 5):
    all_blocks = [''.join(map(str, b)) for b in product(range(U), repeat=m)]
    assert len(all_blocks) == U ** m          # Omega = U^m, verified
    shown = ' '.join(all_blocks) if len(all_blocks) <= 16 else ' '.join(all_blocks[:8]) + ' ...'
    print(f"{m:>3}  {U ** m:>12}   {shown}")

print("\nAnd the m = 1 consistency check:")
print(f"  Omega = U^1 = {U ** 1} = U   ->   the block alphabet is the character alphabet")

---
## 4. Information entropy of a set of $m$-blocks

With the new alphabet defined, Shannon's formula is applied unchanged — only the symbols are
relabelled. Writing $H(\mathcal{X})^{(m)}$ for the entropy of the set of $m$-blocks:

$$H(\mathcal{X})^{(m)} = -\sum_{j=1}^{\Omega} p_j^{(m)} \log_2 p_j^{(m)}
\quad \text{[bits per $m$-block]}$$

where

- the sum now runs to $\Omega$ (the number of distinct $m$-blocks) rather than to $U$;
- $p_j^{(m)}$ is the probability that the $j$-th **$m$-block** occurs — the superscript $(m)$ is
  there purely to remind us these are block probabilities, not single-character ones.

That is the entire generalisation. Nothing about the mathematics of entropy changes; we have only
enlarged the alphabet.

⚠️ **Watch the units.** $H(\mathcal{X})^{(m)}$ is measured in **bits per block**, not bits per
character. A value of 4 bits per block means something very different at $m=2$ than at $m=8$. To
compare across block sizes we will need to divide by $m$ — see Section 6.

### Presenter's Notes — The $m$-block entropy formula
![Board showing the Shannon entropy formula written for m-blocks](Screenshots/Part_3_00_07_00.jpg)

*Screenshot at timestamp 00:07:00*

---

In [ ]:
def block_entropy(sequence, m, s=None, base=2):
    """H(X)^(m) in bits per block, for blocks generated with slide s."""
    bs = make_blocks(sequence, m, s)
    counts = Counter(bs)
    n = len(bs)
    return shannon_entropy([c / n for c in counts.values()], base=base), counts

# A short binary message, analysed at m = 1 and m = 2
msg = [0, 1, 1, 1, 1, 0, 1, 1, 0, 1]   # the Part 2 example message
print(f"message (N = {len(msg)}): {''.join(map(str, msg))}\n")

for m in (1, 2):
    H_m, counts = block_entropy(msg, m)          # non-overlapping, s = m
    Omega = 2 ** m
    print(f"m = {m}   (Omega = 2^{m} = {Omega} possible blocks)")
    print(f"  blocks observed: {[''.join(map(str, b)) for b in make_blocks(msg, m)]}")
    for blk, c in sorted(counts.items()):
        print(f"    p('{''.join(map(str, blk))}') = {c}/{sum(counts.values())} "
              f"= {c / sum(counts.values()):.3f}")
    print(f"  distinct blocks seen: {len(counts)} of {Omega} possible")
    print(f"  H(X)^({m}) = {H_m:.4f} bits per block\n")

---
## 5. Maximum entropy scales linearly with $m$

The maximum entropy of any set is the log of the number of distinct characters in it — attained
when every character is equally likely. For the set of $m$-blocks that count is $\Omega$, so

$$H(\mathcal{X})^{(m)}_{\max} = \log_2 \Omega .$$

Substituting $\Omega = U^{\,m}$ and using $\log(a^b) = b\log(a)$:

$$H(\mathcal{X})^{(m)}_{\max} = \log_2\!\left(U^{\,m}\right) = m \log_2 U$$

But $\log_2 U$ is precisely the maximum entropy of the *single-character* set. So the two are
linked by nothing more complicated than a factor of $m$:

$$\boxed{\;H(\mathcal{X})^{(m)}_{\max} = m \cdot H(\mathcal{X})_{\max}\;}$$

**The maximum information entropy of a set of $m$-blocks is $m$ times that of the set of single
characters.** Doubling the block size doubles the ceiling — which makes sense, since a block of $m$
characters can carry at most $m$ times what one character carries.

### Presenter's Notes — $H(\mathcal{X})^{(m)}_{\max} = m \cdot H(\mathcal{X})_{\max}$
![Board showing the boxed maximum entropy scaling relationship](Screenshots/Part_3_00_11_40.jpg)

*Screenshot at timestamp 00:11:40*

---

In [ ]:
print(f"{'U':>3}  {'m':>3}  {'Omega = U^m':>12}  {'log2(Omega)':>12}  "
      f"{'m*log2(U)':>11}  {'match':>6}")
print("-" * 60)
for U in (2, 4, 26):
    for m in (1, 2, 3, 4):
        Omega = U ** m
        lhs = np.log2(Omega)          # H_max^(m) computed from Omega directly
        rhs = m * np.log2(U)          # H_max^(m) as m * H_max
        print(f"{U:>3}  {m:>3}  {Omega:>12,}  {lhs:>12.4f}  {rhs:>11.4f}  "
              f"{str(np.isclose(lhs, rhs)):>6}")

print("\nThe two columns agree exactly -- H_max^(m) = m * H_max is an identity,")
print("not an approximation.")

---
## 6. Worked example — a binary message at $m = 4$

Take a binary message ($U = 2$) and group it into blocks of four. The possible-state count is

$$\Omega = U^{\,m} = 2^4 = 16$$

so there are 16 distinct 4-blocks available, running from `0000` to `1111` — every permutation of
four bits.

A short message cannot possibly contain all 16. If $N = 20$ characters are split into
non-overlapping 4-blocks we get only $20/4 = 5$ blocks, so at most 5 of the 16 possible states can
appear, and some may repeat. This gap between *possible* and *observed* is the heart of the next
two sections.

### Presenter's Notes — Enumerating the 16 possible 4-blocks
![Board enumerating the sixteen possible four-bit blocks](Screenshots/Part_3_00_19_05.jpg)

*Screenshot at timestamp 00:19:05*

---

In [ ]:
message20 = [0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0]
m = 4
Omega = 2 ** m

blocks4 = make_blocks(message20, m)                      # non-overlapping, s = m
labels = [''.join(map(str, b)) for b in blocks4]
H4, counts4 = block_entropy(message20, m)

print(f"message (N = {len(message20)}): {''.join(map(str, message20))}")
print(f"m = {m},  Omega = 2^{m} = {Omega} possible 4-blocks\n")
print(f"blocks obtained ({len(blocks4)} of them): {labels}\n")

for blk, c in sorted(counts4.items()):
    lbl = ''.join(map(str, blk))
    print(f"  p('{lbl}') = {c}/{len(blocks4)} = {c / len(blocks4):.3f}")

H_max_4 = m * np.log2(2)
print(f"\ndistinct blocks observed : {len(counts4)} of {Omega} possible "
      f"({len(counts4) / Omega:.0%} of the state space)")
print(f"H(X)^(4)                 = {H4:.4f} bits per block")
print(f"H(X)^(4)_max = m*log2(U) = {H_max_4:.4f} bits per block")
print(f"efficiency               = {H4 / H_max_4:.1%} of maximum")

In [ ]:
# All 16 possible 4-blocks, with the ones this message happens to contain marked
observed = {''.join(map(str, b)) for b in blocks4}
all16 = [''.join(map(str, b)) for b in product((0, 1), repeat=4)]

fig, ax = plt.subplots(figsize=(11, 4))
freq = [sum(1 for lb in labels if lb == b) for b in all16]
colors = ['steelblue' if f > 0 else 'lightgrey' for f in freq]
ax.bar(all16, freq, color=colors, edgecolor='white')
ax.set_ylabel('count in message', fontsize=11)
ax.set_xlabel('the $\\Omega = 16$ possible 4-blocks', fontsize=11)
ax.set_title(f'Only {len(observed)} of the {Omega} possible states appear in a 20-character message',
             fontsize=12)
ax.set_yticks(range(0, max(freq) + 1))
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"observed: {sorted(observed)}")
print(f"absent:   {sorted(set(all16) - observed)}")

---
## 7. $\Omega$, microstates, and why the notation was chosen

The choice of $\Omega$ is deliberate. In statistical mechanics, $\Omega$ denotes the number of
**microstates** accessible to a physical system, and Boltzmann's entropy is

$$S = k_B \ln \Omega$$

Set alongside the maximum information entropy of an $m$-block set,

$$H(\mathcal{X})^{(m)}_{\max} = \log_2 \Omega$$

the two are the *same statement* about counting states, differing only in the constant out front
and the base of the logarithm ($k_B$ and $\ln$ for thermodynamics; $1$ and $\log_2$ for bits).

The parallel is exact in structure:

| Information theory | Statistical mechanics |
|---|---|
| $m$-block of characters | microstate of a system |
| $\Omega = U^m$ distinct blocks | $\Omega$ accessible microstates |
| $H_{\max} = \log_2 \Omega$ bits | $S = k_B \ln \Omega$ joules/kelvin |
| all blocks equally likely | equilibrium / equal a priori probability |

A large physical system is built from very many microstates, in the same way a long message is
built from very many possible $m$-blocks — and in both cases the count grows **exponentially** with
system size while the entropy grows only **linearly**. That is the content of
$H^{(m)}_{\max} = m \cdot H_{\max}$: the state count $\Omega = U^m$ explodes, but its logarithm is
merely proportional to $m$.

This correspondence between counting message states and counting physical states is the bridge the
rest of the series is built on.

### Presenter's Notes — Blocks as microstates of a large system
![Board relating m-blocks to microstates of a large physical system](Screenshots/Part_3_00_24_10.jpg)

*Screenshot at timestamp 00:24:10*

---

In [ ]:
# Omega explodes exponentially; its logarithm (the entropy ceiling) grows linearly.
m_vals = np.arange(1, 21)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for U, c in ((2, 'steelblue'), (4, 'darkorange')):
    axes[0].semilogy(m_vals, U ** m_vals.astype(float), 'o-', color=c,
                     markersize=4, label=f'$U = {U}$')
    axes[1].plot(m_vals, m_vals * np.log2(U), 'o-', color=c,
                 markersize=4, label=f'$U = {U}$')

axes[0].set_xlabel('block size $m$', fontsize=11)
axes[0].set_ylabel('$\\Omega = U^m$  (log scale)', fontsize=11)
axes[0].set_title('State count explodes exponentially', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3, which='both')

axes[1].set_xlabel('block size $m$', fontsize=11)
axes[1].set_ylabel('$H^{(m)}_{max} = \\log_2 \\Omega$  (bits)', fontsize=11)
axes[1].set_title('But its logarithm grows only linearly', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"{'m':>3}  {'Omega (U=2)':>16}  {'H_max^(m) [bits]':>17}")
print("-" * 40)
for m in (1, 4, 8, 16, 32, 64):
    print(f"{m:>3}  {2 ** m:>16,}  {m * 1.0:>17.1f}")

---
## 8. Per-character entropy — what blocking actually reveals

$H(\mathcal{X})^{(m)}$ is in bits *per block*, so it grows roughly linearly with $m$ and cannot be
compared across block sizes directly. Dividing by the block length puts every $m$ on the same
footing:

$$h_m = \frac{H(\mathcal{X})^{(m)}}{m} \quad \text{[bits per character]}$$

Now a genuinely interesting question can be asked: **as $m$ increases, what happens to $h_m$?**

The answer depends entirely on whether the characters are independent of one another.

### Case 1 — a memoryless source

If each character is drawn independently, entropies of independent quantities add:

$$H^{(m)} = \underbrace{H^{(1)} + \cdots + H^{(1)}}_{m} = m\,H^{(1)}
\qquad\Longrightarrow\qquad h_m = H^{(1)} \;\text{ for every } m .$$

$h_m$ is **flat**. Blocking reveals nothing, because there is no structure between characters to
find.

### Case 2 — a source with memory

If characters depend on their neighbours, blocks capture correlations that single characters
cannot, and $h_m$ **decreases** with $m$. The single-character entropy then *overstates* the true
information content of the source.

We test both below, using the fully overlapping slide $s=1$ to extract the most blocks from a
given message.

In [ ]:
rng = np.random.default_rng(seed=42)

# --- Case 1: memoryless (i.i.d.) binary source with p(1) = 0.7 ---
p1 = 0.7
iid_seq = rng.choice([0, 1], size=200_000, p=[1 - p1, p1])
H1_theory = shannon_entropy([1 - p1, p1])

# --- Case 2: symmetric Markov source that flips with probability eps ---
def markov_sequence(n, eps, rng):
    """Binary Markov chain flipping state with probability eps each step.

    Symmetric, so the stationary distribution is uniform and H^(1) = 1 bit exactly.
    """
    return np.cumsum(rng.random(n) < eps) % 2

eps = 0.05
markov_seq = markov_sequence(200_000, eps, rng)
h_inf = shannon_entropy([eps, 1 - eps])   # conditional entropy H(X_t+1 | X_t)

print(f"Memoryless source : H^(1) = {H1_theory:.4f} bits/character")
print(f"Markov source     : H^(1) = {entropy_of_message(markov_seq)[0]:.4f} bits/character "
      f"(looks maximal!)")
print(f"                    true rate = {h_inf:.4f} bits/character\n")

print(f"{'m':>3}  {'memoryless h_m':>15}  {'Markov h_m':>12}")
print("-" * 34)
iid_hm, markov_hm = [], []
for m in range(1, 13):
    Hi, _ = block_entropy(iid_seq, m, s=1)
    Hk, _ = block_entropy(markov_seq, m, s=1)
    iid_hm.append(Hi / m)
    markov_hm.append(Hk / m)
    print(f"{m:>3}  {Hi / m:>15.4f}  {Hk / m:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

m_ax = np.arange(1, len(iid_hm) + 1)

axes[0].plot(m_ax, iid_hm, 'o-', color='steelblue', linewidth=2, label='$h_m = H^{(m)}/m$')
axes[0].axhline(H1_theory, color='crimson', linestyle='--', linewidth=1.5,
                label=f'$H^{{(1)}}$ = {H1_theory:.3f} bits')
axes[0].set_title('Memoryless source ($p_1 = 0.7$)\nno structure to find — $h_m$ is flat',
                  fontsize=11)

axes[1].plot(m_ax, markov_hm, 'o-', color='darkorange', linewidth=2, label='$h_m = H^{(m)}/m$')
axes[1].axhline(1.0, color='gray', linestyle=':', linewidth=1.5,
                label='$H^{(1)}$ = 1.000 bits (apparent)')
axes[1].axhline(h_inf, color='crimson', linestyle='--', linewidth=1.5,
                label=f'true rate = {h_inf:.3f} bits')
axes[1].set_title(f'Markov source ($\\epsilon = {eps}$)\n'
                  'blocking exposes the memory — $h_m$ decays', fontsize=11)

for ax in axes:
    ax.set_xlabel('block size $m$', fontsize=11)
    ax.set_ylabel('bits per character', fontsize=11)
    ax.set_ylim(0, 1.08)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. The limit — the entropy rate

The decay on the right-hand plot is completely general. For any **stationary** source, $h_m$ is
non-increasing in $m$ (more context never hurts) and bounded below by zero — so it converges. The
limit is the **entropy rate**:

$$h_\infty = \lim_{m \to \infty} \frac{H(\mathcal{X})^{(m)}}{m}$$

This is the honest answer to "how much information does this source produce per character?", and by
Shannon's source coding theorem it is the hard floor for lossless compression.

### A faster-converging equivalent

Instead of the *average* cost over a block, take the **marginal** cost of one extra character:

$$h_\infty = \lim_{m \to \infty}\left( H^{(m)} - H^{(m-1)} \right)$$

Both limits give the same value, but the marginal form converges far faster. For a first-order
Markov chain we can see why exactly — the block entropy has a closed form:

$$H^{(m)} = H^{(1)} + (m-1)\,h_\infty$$

a **straight line** in $m$. So the marginal form $H^{(m)} - H^{(m-1)}$ equals $h_\infty$ *exactly*
for every $m \geq 2$, landing on the answer immediately, while the average form
$H^{(m)}/m = \frac{H^{(1)} + (m-1)h_\infty}{m}$ keeps dragging the expensive first character into
the average and approaches the limit only as $O(1/m)$.

In [ ]:
H_series = np.array([block_entropy(markov_seq, m, s=1)[0] for m in range(1, 15)])
m_vals = np.arange(1, len(H_series) + 1)

h_average = H_series / m_vals                                  # H^(m) / m
h_marginal = np.diff(np.concatenate([[0.0], H_series]))        # H^(m) - H^(m-1)
h_exact = (1.0 + (m_vals - 1) * h_inf) / m_vals                # closed form for a Markov chain

print(f"{'m':>3}  {'H^(m)':>9}  {'H^(m)/m':>9}  {'H^(m)-H^(m-1)':>15}  {'exact h_m':>10}")
print("-" * 54)
for i, m in enumerate(m_vals):
    print(f"{m:>3}  {H_series[i]:>9.4f}  {h_average[i]:>9.4f}  "
          f"{h_marginal[i]:>15.4f}  {h_exact[i]:>10.4f}")
print(f"\ntrue entropy rate h_inf = {h_inf:.4f} bits/character")

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(m_vals, h_average, 'o-', color='steelblue', linewidth=2, label='$H^{(m)}/m$  (average)')
ax.plot(m_vals, h_marginal, 's-', color='seagreen', linewidth=2,
        label='$H^{(m)} - H^{(m-1)}$  (marginal)')
ax.plot(m_vals, h_exact, 'k--', linewidth=1.5, label='exact $h_m$ (closed form)')
ax.axhline(h_inf, color='crimson', linestyle=':', linewidth=1.5,
           label=f'$h_\\infty$ = {h_inf:.3f} bits')
ax.set_xlabel('block size $m$', fontsize=11)
ax.set_ylabel('bits per character', fontsize=11)
ax.set_title('Two routes to the entropy rate\nthe marginal form lands on it immediately',
             fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Redundancy — reading the gap

The distance between the ceiling and the true rate is **redundancy**, the predictable part of a
message:

$$R = H_{\max} - h_\infty = \log_2 U - h_\infty \quad \text{[bits per character]}$$

usually quoted as a fraction:

$$r = \frac{R}{H_{\max}} = 1 - \frac{h_\infty}{\log_2 U}$$

Redundancy is not waste. It is exactly what makes error correction possible, and exactly what a
compressor removes. A source with $r = 0.5$ can in principle be losslessly compressed to half its
size.

This is why the $m$-block view matters for real data. Natural language, DNA and images all look far
more random character-by-character than they truly are — only blocking makes the structure visible.

In [ ]:
H_max_1 = np.log2(2)          # binary alphabet ceiling: 1 bit per character
R_true = H_max_1 - h_inf

print(f"H_max  (binary alphabet)        = {H_max_1:.4f} bits/character")
print(f"h_1    (measured at m = 1)      = {markov_hm[0]:.4f} bits/character")
print(f"h_inf  (true entropy rate)      = {h_inf:.4f} bits/character")
print()
print(f"Redundancy visible at m = 1     = {H_max_1 - markov_hm[0]:.4f} bits/character "
      f"({(H_max_1 - markov_hm[0]) / H_max_1:.1%})   <- the memory is invisible here")
print(f"Redundancy visible at m = {len(markov_hm):<2}    = {H_max_1 - markov_hm[-1]:.4f} bits/character "
      f"({(H_max_1 - markov_hm[-1]) / H_max_1:.1%})")
print(f"True redundancy                 = {R_true:.4f} bits/character ({R_true / H_max_1:.1%})")
print()
print(f"Best possible lossless compression: down to {h_inf / H_max_1:.1%} of the original size.")

---
## 11. A practical limit — how large can $m$ actually be?

The definition asks for $m \to \infty$, but any real estimate comes from a finite message. Here
$\Omega = U^m$ works against us: the number of blocks whose probabilities we must estimate grows
**exponentially** in $m$, while the number of blocks we actually observe only falls as $N - m + 1$.

Once $\Omega$ approaches the sample size, most possible blocks are never seen at all. A block that
never occurs contributes nothing to the sum, so the estimator cannot see the tail of the
distribution it is measuring, and $H^{(m)}$ comes out **too low**.

Two cautions about demonstrating this honestly:

1. **Bias is an average over realisations, not a property of one sequence.** A single random
   sequence can land either side of the truth. Below we average over many independent
   realisations, which is what makes the systematic effect visible.
2. **We need a known answer to measure error against.** The Markov chain supplies one:
   $h_m = \frac{H^{(1)} + (m-1)h_\infty}{m}$ exactly.

The rule of thumb: keep $\Omega = U^m \ll N$, ideally by a factor of 10–100.

In [ ]:
def h_block_fast(seq, m):
    """Per-character block entropy for a binary sequence (s = 1), via integer-coded windows.

    Equivalent to block_entropy(seq, m, s=1)[0] / m, but fast enough to average
    over many realisations.
    """
    w = np.lib.stride_tricks.sliding_window_view(np.asarray(seq), m)
    codes = w @ (1 << np.arange(m - 1, -1, -1))
    counts = np.bincount(codes)
    counts = counts[counts > 0]
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum() / m)

# sanity check against the readable implementation
_chk = block_entropy(markov_seq[:5000], 6, s=1)[0] / 6
assert np.isclose(_chk, h_block_fast(markov_seq[:5000], 6)), "implementations disagree"
print(f"fast and readable implementations agree: {_chk:.6f}\n")

ms = np.arange(1, 15)
exact = (1.0 + (ms - 1) * h_inf) / ms          # known truth for this source
REALISATIONS = 30
bias_rng = np.random.default_rng(seed=7)

errors = {}
for N_s in (1_000, 5_000, 50_000):
    acc = np.zeros(len(ms))
    for _ in range(REALISATIONS):
        seq = markov_sequence(N_s, eps, bias_rng)
        acc += [h_block_fast(seq, m) for m in ms]
    errors[N_s] = acc / REALISATIONS - exact

print(f"Mean signed error (estimate - exact), averaged over {REALISATIONS} realisations:")
print(f"{'m':>3}  {'2^m':>9}  " + "".join(f"{('N=' + format(n, ',')):>12}" for n in errors))
print("-" * (14 + 12 * len(errors)))
for i, m in enumerate(ms):
    print(f"{m:>3}  {2 ** m:>9,}  " + "".join(f"{errors[n][i]:>+12.4f}" for n in errors))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for (N_s, err), c in zip(errors.items(), ['indianred', 'darkorange', 'steelblue']):
    ax.plot(ms, err, 'o-', color=c, linewidth=2, markersize=4, label=f'$N = {N_s:,}$')
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('block size $m$', fontsize=11)
ax.set_ylabel('mean error in $h_m$  (bits per character)', fontsize=11)
ax.set_title(f'Finite-sample bias, averaged over {REALISATIONS} realisations\n'
             'always negative — and worse as $m$ grows or $N$ shrinks', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

worst = errors[1_000]
print(f"At N = 1,000 the estimate is too low at every m, by up to "
      f"{abs(worst.min()):.4f} bits/character.")
print(f"At N = 50,000 the bias has largely vanished: worst case "
      f"{abs(errors[50_000]).max():.4f} bits/character.")

---
## Summary

| Quantity | Definition | Units |
|---|---|---|
| Block size | $m$ — single characters per block | characters |
| Slide | $s$, with $1 \leq s \leq m$ | characters |
| Distinct $m$-blocks | $\Omega = U^{\,m}$ | blocks |
| Block entropy | $H(\mathcal{X})^{(m)} = -\sum_{j=1}^{\Omega} p_j^{(m)}\log_2 p_j^{(m)}$ | bits per block |
| Maximum block entropy | $H(\mathcal{X})^{(m)}_{\max} = \log_2\Omega = m\cdot H(\mathcal{X})_{\max}$ | bits per block |
| Per-character entropy | $h_m = H(\mathcal{X})^{(m)}/m$ | bits per character |
| Entropy rate | $h_\infty = \lim_{m\to\infty} H^{(m)}/m = \lim_{m\to\infty}\left(H^{(m)} - H^{(m-1)}\right)$ | bits per character |
| Redundancy | $R = \log_2 U - h_\infty$ | bits per character |

### Key takeaways

1. **Blocking generalises the formula without changing it.** Treat each $m$-block as a character of
   a larger alphabet and apply the identical Shannon sum — only the upper limit changes from $U$ to
   $\Omega$.
2. **$\Omega = U^{\,m}$** is the central counting relationship, and it recovers $\Omega = U$ at
   $m=1$ — Part 2 was this lecture's special case all along.
3. **$H^{(m)}_{\max} = m\cdot H_{\max}$.** The state count grows exponentially in $m$; its
   logarithm, the entropy ceiling, grows only linearly.
4. **$\Omega$ is Boltzmann's symbol on purpose.** $H_{\max} = \log_2\Omega$ and $S = k_B\ln\Omega$
   are the same statement about counting states.
5. **Per-character entropy $h_m = H^{(m)}/m$ is what to compare across block sizes.** Flat for a
   memoryless source; decreasing for a source with memory.
6. **The limit is the entropy rate $h_\infty$** — the true information per character and the floor
   for lossless compression. Our Markov source *looked* like a full 1 bit per character while
   actually producing about $0.29$.
7. **Finite data caps how far $m$ can go.** Because $\Omega = U^m$ grows exponentially, block
   entropy estimates are biased low once $\Omega$ approaches $N$.

### Where this leads

Counting the states of a message and counting the microstates of a physical system turn out to be
the same operation. That equivalence — information entropy and thermodynamic entropy as two
readings of $\log \Omega$ — is the foundation the rest of this series builds on.

---

> 🧭 **Navigation** · [← Part 2: Information Entropy — Worked Examples](../Video_02/Information_Entropy_Examples.ipynb) · **Part 3 (current)** · *Part 4: coming soon*